# LeWM embeddings in 3-D (UMAP)

For every entry in `RUNS`: encode all states of a dataset with a trained LeWM, reduce the 192-d embeddings to 3-D with UMAP
and show them as an interactive plot. Hovering over a point shows that state's grid on the right.

UMAP distorts distances and cluster sizes: use the plots for intuition, not as a measurement.
Kernel: the project's `.venv` (Python 3.12). After changing the settings, run all cells again (Run All).

In [ ]:
import io
import os

os.environ.setdefault("STABLEWM_HOME", "/home/lukas/TU_Dresden/master_thesis/data/stable_worldmodel")  # the notebook kernel may not see the shell's export

import ipywidgets as widgets
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch
import umap
from IPython.display import display
from omegaconf import OmegaConf
from PIL import Image

from master_thesis.evaluation.arc_dataset_metric_visualization import load_columns  # same column loading as the dataset metrics
from master_thesis.evaluation.arc_policies import load_lewm  # same model loading as the planning evaluation
from master_thesis.paths import evaluation_dir, model_dir
from master_thesis.training.lewm import ArcGridToPixels  # same preprocessing as in training

## Settings

In [ ]:
GAME = "ls20"
RUNS = [  # (model folder in models/lewm, dataset whose states are embedded, weights file inside the model folder)
    ("ls20_l1-human-goose_150k_0918-0120", "l1_human-goose", "weights.pt"),
    ("ls20_l2-human-goose_150k_0918-0120", "l2_human-goose", "weights.pt"),
    # ("ls20_l1-human-goose_150k_0918-0120", "l1_human-goose", "checkpoints/weights_step110000.pt"),  # an intermediate checkpoint
]
COLOR_BY = "progress"  # progress (0 = episode start, 1 = end) | level | completed | action
EPISODE = 0  # highlight this episode as a black path (episode index as in the GIF file names); None = no path
N_NEIGHBORS = 15  # UMAP: small = fine local structure (one level), large = global structure (many levels)
SEED = 0  # UMAP seed -> the same layout every time
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## Functions

In [ ]:
PALETTE = (ArcGridToPixels(64).palette.numpy() * 255).round().astype(np.uint8)  # [16, 3] ARC colour index -> RGB


def load_states(dataset):
    """All states of datasets/<GAME>/<dataset>.lance as a table (one row per state) plus the raw grids."""
    grids, actions, levels, successes, episodes = load_columns(GAME, dataset)
    lengths = np.array([length for _, length in episodes])  # states per episode
    starts = np.array([start for start, _ in episodes])  # first row of every episode
    episode = np.repeat(np.arange(len(episodes)), lengths)  # episode index of every row (episodes are stored one after another)
    step = np.arange(len(grids)) - starts[episode]  # position of the state inside its episode
    table = pd.DataFrame({
        "row": np.arange(len(grids)),  # row in grids, used by the hover
        "episode": episode,
        "step": step,
        "progress": step / np.maximum(lengths[episode] - 1, 1),  # 0 = first state, 1 = last state
        "level": levels,
        "completed": successes,  # did the episode complete its level?
        "action": actions,  # action taken in this state (0 = last state)
    })
    return table, grids


@torch.no_grad()
def embed(run_name, weights, grids, batch_size=256):
    """Encode every grid with a trained LeWM: [states, 4096] -> [states, 192]."""
    model = load_lewm(run_name, weights=weights, device=DEVICE)  # architecture + trained weights
    img_size = int(OmegaConf.load(model_dir("lewm", run_name) / "train_config.yaml").img_size)  # image size used in training
    to_pixels = ArcGridToPixels(img_size)  # same preprocessing as in training
    embeddings = []
    for start in range(0, len(grids), batch_size):  # batches keep GPU memory small
        pixels = to_pixels(torch.as_tensor(grids[start:start + batch_size], device=DEVICE))  # [B, 3, img, img]
        embeddings.append(model.encode({"pixels": pixels.unsqueeze(1)})["emb"][:, 0].float().cpu().numpy())  # encode expects [B, time, ...] -> [B, 192]
    del model  # free GPU memory before the next run
    torch.cuda.empty_cache()
    return np.concatenate(embeddings)


def umap_layout(embeddings, dimensions=3):
    """Reduce the embeddings to `dimensions` axes with UMAP."""
    reducer = umap.UMAP(n_components=dimensions, n_neighbors=N_NEIGHBORS, min_dist=0.1, init="pca", random_state=SEED)  # PCA start: the default spectral start fails on the many identical states
    return reducer.fit_transform(embeddings)


def grid_png(grid, scale=4):
    """PNG bytes of one 64x64 grid, enlarged `scale` times with sharp pixels."""
    image = Image.fromarray(PALETTE[np.asarray(grid).reshape(64, 64)]).resize(
        (64 * scale, 64 * scale), Image.NEAREST)
    buffer = io.BytesIO()
    image.save(buffer, format="PNG")
    return buffer.getvalue()

def interactive_3d(table, grids, title, html_path=None):
    """3-D scatter of the UMAP layout; hovering a point shows its grid next to the plot."""
    colour = table[COLOR_BY] if COLOR_BY == "progress" else table[COLOR_BY].astype(str)  # progress = continuous colours, everything else = one colour per value
    figure = px.scatter_3d(table.assign(colour=colour), x="x", y="y", z="z", color="colour", labels={"colour": COLOR_BY},
                           custom_data=["row"], hover_data=["episode", "step", "level", "action", "completed"], title=title)
    figure.update_traces(marker_size=2)  # small points

    if EPISODE is not None:  # draw one episode in order as a black path
        path = table[table["episode"] == EPISODE]
        figure.add_trace(go.Scatter3d(x=path["x"], y=path["y"], z=path["z"], mode="lines+markers", line={"color": "black", "width": 3},
                                      marker={"size": 3, "color": "black"}, customdata=path[["row"]].to_numpy(), name=f"episode {EPISODE}"))

    if html_path is not None:
        figure.write_html(html_path)  # interactive file that opens in any browser (without the grid panel)

    widget = go.FigureWidget(figure)  # interactive copy that can call Python on hover
    widget.update_layout(width=750, height=650)
    image = widgets.Image(format="png", width=256, height=256)  # the hovered state
    caption = widgets.HTML("hover over a point")  # text below the image

    def show_state(trace, points, state):  # called by plotly when the mouse is over a point
        if not points.point_inds:  # the mouse is not over a point of this trace
            return
        row = int(trace.customdata[points.point_inds[0]][0])  # row of the hovered state
        image.value = grid_png(grids[row])
        info = table.iloc[row]
        caption.value = f"episode {info.episode} · step {info.step} · level {info.level} · action {info.action} · {'completed' if info.completed else 'not completed'}"

    for trace in widget.data:  # px makes one trace per colour, plus the episode path
        trace.on_hover(show_state)
    return widgets.HBox([widget, widgets.VBox([image, caption])])  # plot left, state right

## Run

In [ ]:
results = {}  # (run, dataset, weights) -> (table, embeddings), also used by the optional PNG below
for run_name, dataset, weights in RUNS:
    table, grids = load_states(dataset)  # states + labels
    embeddings = embed(run_name, weights, grids)  # [states, 192]
    table[["x", "y", "z"]] = umap_layout(embeddings)  # 3-D coordinates
    results[(run_name, dataset, weights)] = (table, embeddings)

    print(f"{run_name} ({weights}) on {dataset}: {len(table)} states, {table['episode'].nunique()} episodes")
    html = evaluation_dir(GAME) / f"{dataset}_umap3d_{run_name}.html"  # file name starts with the dataset
    display(interactive_3d(table, grids, f"{run_name} on {dataset}", html_path=html))

## Optional: 2-D overview as PNG

Not called by default. Saves one image per run with four colourings (level, completed, progress, action) to `evaluation/<game>/`.

In [ ]:
import matplotlib.pyplot as plt


def save_overview_png(run_name, dataset, weights="weights.pt"):
    """2-D UMAP of one run with four colourings, saved as <dataset>_umap2d_<run>.png."""
    table, embeddings = results[(run_name, dataset, weights)]
    xy = umap_layout(embeddings, dimensions=2)  # separate 2-D layout (not a projection of the 3-D one)
    figure, axes = plt.subplots(1, 4, figsize=(20, 4.5))
    for axis, column in zip(axes, ["level", "completed", "progress", "action"]):
        points = axis.scatter(xy[:, 0], xy[:, 1], c=table[column].astype(float), cmap="viridis" if column == "progress" else "tab10", s=2)
        axis.set_title(column)
        axis.set_xticks([])
        axis.set_yticks([])
        figure.colorbar(points, ax=axis, fraction=0.046)
    figure.suptitle(f"{run_name} ({weights}) on {dataset}")
    path = evaluation_dir(GAME) / f"{dataset}_umap2d_{run_name}.png"
    figure.savefig(path, dpi=120, bbox_inches="tight")
    plt.close(figure)  # do not show it in the notebook
    return path


# save_overview_png("ls20_l1-human-goose_150k_0918-0120", "l1_human-goose")